# 03 - Test ALL THREE trained models (v1, v2, v3): predict + re-score

**No training happens in this notebook.** It only loads the models you already trained (saved in Google Drive) and (a) predicts on one big paragraph, (b) re-scores each model on test, extra_test and pattern_test.

Before you start: Runtime > Change runtime type > **T4 GPU**. Then run the cells top to bottom.

| Name | Drive folder (`MyDrive/ner_results/`) | Local folder after copy |
|---|---|---|
| v1 | `xlmr_base` | `outputs/xlmr_base/best` |
| v2 | `xlmr_v2` | `outputs/xlmr_v2/best` |
| v3 | `xlmr_v3` | `outputs/xlmr_v3/best` |

## 1. Check the GPU
`nvidia-smi` is the NVIDIA driver status tool (not a download). It must show a Tesla T4.

In [ ]:
!nvidia-smi

## 2. Install libraries (about 1 minute)

In [ ]:
!pip -q install -U transformers accelerate

## 3. Upload `colab_bundle.zip` (from `outputs/` on your PC) and unzip
Click *Choose files* and select the zip. The zip already contains a top-level folder called `ner`, so it is unzipped as it is (no `-d`).

In [ ]:
from google.colab import files
up = files.upload()
!rm -rf ner
!unzip -oq colab_bundle.zip
%cd ner
!ls scripts data/dataset

## 4. Copy the three trained models from Google Drive
A pop-up asks permission to access Drive: allow it. Each `best` folder is about 1.1 GB, so this takes a few minutes. The last line lists the files to prove all three are there.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SRC = '/content/drive/MyDrive/ner_results'
for run in ['xlmr_base', 'xlmr_v2', 'xlmr_v3']:
    !mkdir -p outputs/{run}
    !cp -r {SRC}/{run}/best outputs/{run}/
!ls outputs/xlmr_base/best outputs/xlmr_v2/best outputs/xlmr_v3/best

## 5. Record library versions (for the report)

In [ ]:
!pip list 2>/dev/null | grep -E "^(torch|transformers|accelerate|numpy) "

## 6. The big test paragraph
One paragraph (one line, because `predict.py` treats each line as one input; long text is cut into 100-word windows automatically). It contains all 20 entity types. **All phone numbers and CNICs are fake (synthetic).**

Feel free to edit the text, or paste your own paragraph between the quotes.

In [ ]:
PARAGRAPH = (
    "On 14 August 2023, Prime Minister Anwaar-ul-Haq Kakar addressed a large gathering at Jinnah Convention Centre in Islamabad, "
    "where he praised the Pakistan Army and the Election Commission of Pakistan for their role in the upcoming general elections. "
    "Imran Khan, the chairman of PTI, had earlier told reporters in Lahore that his party would challenge the delimitation in the Lahore High Court, "
    "while Bilawal Bhutto Zardari of the Pakistan Peoples Party and Nawaz Sharif of PML-N held separate rallies in Karachi and Faisalabad. "
    "The Supreme Court of Pakistan, hearing a petition under the Elections Act 2017 and Article 184(3) of the Constitution, adjourned the case until 9:30 am on Monday. "
    "Dawn and Geo News reported that the Sindh Assembly and the Punjab government had approved Rs 25 billion for flood relief, about 3.5 percent of the provincial budget, "
    "and that the World Bank and the Edhi Foundation would send aid to Balochistan and Khyber Pakhtunkhwa. "
    "Students of NED University of Engineering and Technology and the University of Karachi staged a protest outside the Karachi Press Club at 4 pm on 2 October, "
    "demanding that the Higher Education Commission reduce fees by 20%. "
    "Meanwhile Babar Azam led Pakistan to victory over India at the Asia Cup in Colombo, and the Karachi Kings announced a new sponsor ahead of the Pakistan Super League. "
    "Officials from the Federal Investigation Agency said that anyone with information could call 0300-1234567 or 051-2345678, "
    "and that applicants must quote their CNIC 42101-9876543-5 when contacting the National Database and Registration Authority in Peshawar, Quetta or Gilgit."
)
open('paragraph.txt', 'w', encoding='utf-8').write(PARAGRAPH + '\n')
print(len(PARAGRAPH.split()), 'words saved to paragraph.txt')

## 7. Predict with all three models
Each model prints the entities it found and saves `predictions_v1/v2/v3.jsonl` (with character offsets, tokens and BIO tags).

In [ ]:
MODELS = {'v1': 'outputs/xlmr_base/best', 'v2': 'outputs/xlmr_v2/best', 'v3': 'outputs/xlmr_v3/best'}
for name, path in MODELS.items():
    print('\n' + '=' * 70 + f'\n MODEL {name}  ({path})\n' + '=' * 70)
    !python scripts/predict.py --model {path} --input paragraph.txt --out predictions_{name}.jsonl 2>&1 | grep -v "Loading weights"

## 8. Side-by-side comparison
For every entity found by at least one model, shows which models agree. `v1 v2 v3` columns show the label each model gave (`-` = not found). Rows where the models disagree are marked `<<`, they are the interesting ones to read.

In [ ]:
import json
found = {}   # (start, end, text) -> {model: label}
for name in MODELS:
    for line in open(f'predictions_{name}.jsonl', encoding='utf-8'):
        d = json.loads(line)
        for e in d['entities']:
            found.setdefault((e['start'], e['end'], e['text']), {})[name] = e['label']
short = lambda s: s[:16] if s else '-'
print(f"{'TEXT':42} {'v1':16} {'v2':16} {'v3':16}")
agree = 0
for (s, e, t), lab in sorted(found.items()):
    same = len(lab) == 3 and len(set(lab.values())) == 1
    agree += same
    print(f"{t[:41]:42} {short(lab.get('v1')):16} {short(lab.get('v2')):16} {short(lab.get('v3')):16} {'' if same else '<<'}")
print(f'\n{agree} of {len(found)} entities identical in all three models')
for name in MODELS:
    print(name, 'found', sum(1 for v in found.values() if name in v), 'entities')

## 9. Re-score all three models (no training, about 2 minutes each)
`--eval_only` loads a saved model and scores it on test, extra_test and pattern_test. Results go to `outputs/eval_v1`, `eval_v2`, `eval_v3`.

In [ ]:
for name, path in MODELS.items():
    print('\n' + '#' * 70 + f'\n SCORING {name}\n' + '#' * 70)
    !python scripts/train_ner.py --eval_only {path} --out outputs/eval_{name} 2>&1 | grep -v "Loading weights"

## 10. Download everything
Gets one zip with the three eval folders and the three prediction files. Unzip it into `trained_reports/` on your PC (folders `eval_v1`, `eval_v2`, `eval_v3`) and tell me.

In [ ]:
!zip -rq eval_all.zip outputs/eval_v1 outputs/eval_v2 outputs/eval_v3 predictions_v1.jsonl predictions_v2.jsonl predictions_v3.jsonl paragraph.txt
from google.colab import files
files.download('eval_all.zip')